## GARCH Implementation

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
from arch import arch_model
import sys
from datetime import date

In [2]:
tickers = ['SPY', 'XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLU', 'XLV', 'XLY']

Load returns for training data from ``yfinance``

In [ ]:
training_data = {}
for ticker in tickers:
    # Download and wrangle data
    train_df = yf.download(
        ticker, start = '1994-12-30', end = date.today().strftime("%Y-%m-%d"), auto_adjust=False
    ).reset_index()
    train_df = train_df.droplevel(level=1, axis=1)
    train_df.columns = train_df.columns.str.lower().str.replace(' ', '_')
    train_df.rename(columns = {'date':'trade_date'}, inplace = True)
    train_df = train_df[['trade_date', 'close']].copy()
    train_df.insert(0, 'ticker', ticker)

    # Calculate daily returns (using arithmetic like Sepp)
    train_df['dly_ret'] = (train_df['close'] / train_df['close'].shift(1)) - 1
    train_df = train_df.dropna().reset_index(drop=True)

    # Convert to percentage
    per_returns = 100 * train_df['dly_ret']
    per_returns.index = train_df['trade_date']

    training_data[ticker] = train_df, per_returns

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Forecasting loop (results are already saved to CSVs, run again to update)

In [ ]:
for ticker in training_data:
    print(f'Processing {ticker}')
    train_df, per_returns = training_data[ticker]

    # Instantiate model object
    model = arch_model(per_returns, vol='Garch', p=1, o=0, q=1, dist='normal')
    print(f"Train start: {train_df['trade_date'].iloc[0]}")
    fcast_start = train_df.query('trade_date == "2004-12-31"').index[0]
    fcast_end = train_df.shape[0] - 1

    forecasts = {}
    # Rolling window is length from data start to 2005 (7-10 years depending on ETF)
    for ix in range(fcast_start, fcast_end):
        print(f'\rProcessing index {ix} of {fcast_end}', end = '')
        # Fit model and generate forecast
        result = model.fit(first_obs = (ix - fcast_start), last_obs = ix, disp = 'off')
        temp = result.forecast(horizon = 5, reindex = True).variance
        fcast = temp.iloc[ix - 1]
        forecasts[fcast.name] = fcast
    print()
    forecast_df = pd.DataFrame(pd.DataFrame(forecasts).T)
    forecast_df = forecast_df.reset_index().rename(columns = {'index':'trade_date'})
    forecast_df

    # writing variance estimates to csv-file
    forecast_df.to_csv(f'../data/garch_forecasts/{ticker}_variance_forecast.csv', index = False)

C:\Users\rahul\AppData\Local\Temp\ipykernel_19672\922520189.py:6: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  fcast_start = train_df.query('trade_date == "2004-12-31"').index[0]


Processing SPY
Train start: 1995-01-03 00:00:00
Processing index 7739 of 7740
Processing XLB
Train start: 1998-12-23 00:00:00
Processing index 1513 of 6735

C:\Users\rahul\AppData\Local\Temp\ipykernel_19672\922520189.py:6: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  fcast_start = train_df.query('trade_date == "2004-12-31"').index[0]


Processing index 6734 of 6735
Processing XLE
Train start: 1998-12-23 00:00:00
Processing index 1513 of 6735

C:\Users\rahul\AppData\Local\Temp\ipykernel_19672\922520189.py:6: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  fcast_start = train_df.query('trade_date == "2004-12-31"').index[0]


...

Processing index 6734 of 6735
Processing XLV
Train start: 1998-12-23 00:00:00
Processing index 1513 of 6735

C:\Users\rahul\AppData\Local\Temp\ipykernel_19672\922520189.py:6: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  fcast_start = train_df.query('trade_date == "2004-12-31"').index[0]


Processing index 6734 of 6735
Processing XLY
Train start: 1998-12-23 00:00:00


C:\Users\rahul\AppData\Local\Temp\ipykernel_19672\922520189.py:6: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  fcast_start = train_df.query('trade_date == "2004-12-31"').index[0]


Processing index 6734 of 6735
